# Final RFS Prediction (Regression)

In [1]:
import pandas as pd
import numpy as np
import joblib
import csv

In [2]:
dataset_x_all = pd.read_csv("../datasets/FinalTestDataSet2025.csv")
dataset_x_all

,ID,Age,ER,PgR,HER2,TrippleNegative,ChemoGrade,Proliferation,HistologyType,LNStatus,...,original_glszm_SmallAreaHighGrayLevelEmphasis,original_glszm_SmallAreaLowGrayLevelEmphasis,original_glszm_ZoneEntropy,original_glszm_ZonePercentage,original_glszm_ZoneVariance,original_ngtdm_Busyness,original_ngtdm_Coarseness,original_ngtdm_Complexity,original_ngtdm_Contrast,original_ngtdm_Strength
0,TRG002219,47.0,1,1,0,0,3,2,1,1,...,0.498350,0.498350,3.144594,0.003447,8.257693e+06,150.048587,0.001753,0.037110,0.001369,0.001513
1,TRG002222,41.0,1,1,0,0,3,2,1,0,...,0.622381,0.622381,2.061654,0.006535,1.568442e+06,26.484938,0.009649,0.019352,0.000321,0.008285
2,TRG002223,53.0,0,0,0,1,2,1,1,1,...,0.412482,0.412482,3.440353,0.005391,2.656925e+06,174.606929,0.001594,0.075152,0.005255,0.001444
3,TRG002235,46.0,1,1,0,0,2,1,1,1,...,0.378333,0.378333,3.531715,0.007102,1.714787e+06,96.787378,0.002772,0.053377,0.002666,0.002406
4,TRG002240,39.0,0,0,1,0,2,2,1,1,...,0.524767,0.524767,2.186214,0.007896,5.104793e+05,12.789071,0.020072,0.023140,0.000463,0.017172
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
128,TRG002953,62.3,0,0,1,0,2,1,1,0,...,0.550099,0.550097,2.632664,0.004373,1.583970e+06,91.084710,0.002972,0.042656,0.002168,0.002828
129,TRG002955,46.1,0,0,0,1,3,3,1,1,...,0.439568,0.439568,3.056046,0.001339,1.671271e+07,79.989003,0.003282,0.024716,0.000812,0.003078
130,TRG002956,44.3,0,0,0,1,3,2,1,0,...,0.450758,0.450758,2.820884,0.003774,4.493563e+06,144.506524,0.001912,0.053334,0.003453,0.001759
131,TRG002957,37.1,0,0,0,1,3,3,1,1,...,0.384860,0.384860,3.680654,0.003699,4.054944e+06,250.108074,0.001168,0.091763,0.008428,0.001107


In [3]:
ids = dataset_x_all['ID']
dataset_x_all = dataset_x_all.drop('ID', axis = 1)

In [4]:
dataset_x_all['original_shape_VoxelVolume'] = dataset_x_all['original_shape_VoxelVolume'].astype(np.float64)

categorical_names = dataset_x_all.select_dtypes(np.int64).columns
continuous_names = dataset_x_all.select_dtypes(np.float64).columns

dataset_x_all = dataset_x_all.replace([999], np.nan)

In [5]:
missing_values = dataset_x_all.isnull().sum()
missing_values = missing_values[missing_values > 0]
missing_values

PgR               1
ChemoGrade        1
Proliferation     1
HistologyType     2
LNStatus          3
TumourStage       1
Gene             20
dtype: int64

In [6]:
Xs = dataset_x_all.copy()
Xs

,Age,ER,PgR,HER2,TrippleNegative,ChemoGrade,Proliferation,HistologyType,LNStatus,TumourStage,...,original_glszm_SmallAreaHighGrayLevelEmphasis,original_glszm_SmallAreaLowGrayLevelEmphasis,original_glszm_ZoneEntropy,original_glszm_ZonePercentage,original_glszm_ZoneVariance,original_ngtdm_Busyness,original_ngtdm_Coarseness,original_ngtdm_Complexity,original_ngtdm_Contrast,original_ngtdm_Strength
0,47.0,1,1.0,0,0,3.0,2.0,1.0,1.0,2.0,...,0.498350,0.498350,3.144594,0.003447,8.257693e+06,150.048587,0.001753,0.037110,0.001369,0.001513
1,41.0,1,1.0,0,0,3.0,2.0,1.0,0.0,2.0,...,0.622381,0.622381,2.061654,0.006535,1.568442e+06,26.484938,0.009649,0.019352,0.000321,0.008285
2,53.0,0,0.0,0,1,2.0,1.0,1.0,1.0,2.0,...,0.412482,0.412482,3.440353,0.005391,2.656925e+06,174.606929,0.001594,0.075152,0.005255,0.001444
3,46.0,1,1.0,0,0,2.0,1.0,1.0,1.0,3.0,...,0.378333,0.378333,3.531715,0.007102,1.714787e+06,96.787378,0.002772,0.053377,0.002666,0.002406
4,39.0,0,0.0,1,0,2.0,2.0,1.0,1.0,2.0,...,0.524767,0.524767,2.186214,0.007896,5.104793e+05,12.789071,0.020072,0.023140,0.000463,0.017172
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
128,62.3,0,0.0,1,0,2.0,1.0,1.0,0.0,3.0,...,0.550099,0.550097,2.632664,0.004373,1.583970e+06,91.084710,0.002972,0.042656,0.002168,0.002828
129,46.1,0,0.0,0,1,3.0,3.0,1.0,1.0,4.0,...,0.439568,0.439568,3.056046,0.001339,1.671271e+07,79.989003,0.003282,0.024716,0.000812,0.003078
130,44.3,0,0.0,0,1,3.0,2.0,1.0,0.0,2.0,...,0.450758,0.450758,2.820884,0.003774,4.493563e+06,144.506524,0.001912,0.053334,0.003453,0.001759
131,37.1,0,0.0,0,1,3.0,3.0,1.0,1.0,3.0,...,0.384860,0.384860,3.680654,0.003699,4.054944e+06,250.108074,0.001168,0.091763,0.008428,0.001107


### Data Preprocessing
- Mode Imputation for Categorical Features
- Median Imputation and Standardisation for Continuous Features

In [7]:
mode_imputer = joblib.load('../exports/rfs_mode_imputer.pkl')
median_imputer = joblib.load('../exports/rfs_median_imputer.pkl')
standard_scaler = joblib.load('../exports/rfs_standard_scaler.pkl')

In [8]:
Xs[categorical_names] = mode_imputer.transform(Xs[categorical_names])
Xs[continuous_names] = median_imputer.transform(Xs[continuous_names])

In [9]:
Xs[continuous_names] = standard_scaler.transform(Xs[continuous_names])

In [10]:
Xs[categorical_names] = Xs[categorical_names].astype(np.int64)
Xs

,Age,ER,PgR,HER2,TrippleNegative,ChemoGrade,Proliferation,HistologyType,LNStatus,TumourStage,...,original_glszm_SmallAreaHighGrayLevelEmphasis,original_glszm_SmallAreaLowGrayLevelEmphasis,original_glszm_ZoneEntropy,original_glszm_ZonePercentage,original_glszm_ZoneVariance,original_ngtdm_Busyness,original_ngtdm_Coarseness,original_ngtdm_Complexity,original_ngtdm_Contrast,original_ngtdm_Strength
0,-0.436849,1,1,0,0,3,2,1,1,2,...,0.650177,0.710213,0.527163,0.077949,-0.074526,-0.036565,-0.179605,-0.424803,-0.548869,-0.237184
1,-0.991036,1,1,0,0,3,2,1,0,2,...,1.404233,1.494846,-0.880076,1.330185,-0.083016,-0.142722,-0.179605,-0.804680,-0.673844,-0.183421
2,0.117337,0,0,0,1,2,1,1,1,2,...,0.128133,0.167001,0.911490,0.866326,-0.081635,-0.015467,-0.179606,0.388974,-0.085507,-0.237737
3,-0.529214,1,1,0,0,2,1,1,1,3,...,-0.079478,-0.049028,1.030213,1.559834,-0.082830,-0.082323,-0.179605,-0.076842,-0.394205,-0.230098
4,-1.175765,0,0,1,0,2,2,1,1,2,...,0.810782,0.877331,-0.718215,1.881947,-0.084359,-0.154488,-0.179605,-0.723657,-0.656971,-0.112864
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
128,0.976326,0,0,1,0,2,1,1,0,3,...,0.964789,1.037572,-0.138070,0.453494,-0.082996,-0.087223,-0.179605,-0.306181,-0.453598,-0.226750
129,-0.519977,0,0,0,1,3,3,1,1,4,...,0.292806,0.338351,0.412098,-0.777238,-0.063794,-0.096755,-0.179605,-0.689936,-0.615289,-0.224760
130,-0.686233,0,0,0,1,3,2,1,0,2,...,0.360834,0.409138,0.106515,0.210389,-0.079303,-0.041327,-0.179605,-0.077756,-0.300399,-0.235232
131,-1.351257,0,0,0,1,3,3,1,1,3,...,-0.039800,-0.007742,1.223753,0.179801,-0.079860,0.049398,-0.179606,0.744304,0.292965,-0.240413


### Feature Selection
- Selected using Feature Importances from a Random Forest Regressor
- ER, HER2, Gene were retained since they were important features according to specification

In [11]:
selected_features = joblib.load('../exports/rfs_selected_features.pkl')
selected_features

['original_glszm_ZoneEntropy',
 'original_shape_MajorAxisLength',
 'original_shape_Elongation',
 'Age',
 'original_shape_Flatness',
 'original_firstorder_Skewness',
 'original_firstorder_90Percentile',
 'original_glszm_SizeZoneNonUniformityNormalized',
 'original_firstorder_Range',
 'original_glrlm_RunLengthNonUniformityNormalized',
 'original_firstorder_Kurtosis',
 'original_firstorder_Maximum',
 'original_glrlm_RunEntropy',
 'original_firstorder_Minimum',
 'original_firstorder_Variance',
 'original_glrlm_RunLengthNonUniformity',
 'original_shape_Sphericity',
 'TumourStage',
 'original_shape_Maximum3DDiameter',
 'original_shape_Maximum2DDiameterRow',
 'original_glszm_SmallAreaHighGrayLevelEmphasis',
 'original_firstorder_RootMeanSquared',
 'original_shape_LeastAxisLength',
 'original_glcm_Imc1',
 'original_shape_Maximum2DDiameterColumn',
 'ER',
 'HER2',
 'Gene']

In [12]:
Xs = Xs[selected_features]
Xs

,original_glszm_ZoneEntropy,original_shape_MajorAxisLength,original_shape_Elongation,Age,original_shape_Flatness,original_firstorder_Skewness,original_firstorder_90Percentile,original_glszm_SizeZoneNonUniformityNormalized,original_firstorder_Range,original_glrlm_RunLengthNonUniformityNormalized,...,original_shape_Maximum3DDiameter,original_shape_Maximum2DDiameterRow,original_glszm_SmallAreaHighGrayLevelEmphasis,original_firstorder_RootMeanSquared,original_shape_LeastAxisLength,original_glcm_Imc1,original_shape_Maximum2DDiameterColumn,ER,HER2,Gene
0,0.527163,-0.078897,0.545507,-0.436849,0.490215,0.417851,-0.817105,-0.011221,-0.323090,-0.357218,...,0.019042,0.303096,0.650177,-0.964763,0.480129,0.469386,-0.086732,1,0,1
1,-0.880076,-0.567060,0.812303,-0.991036,1.432086,-0.380669,-0.495912,0.939939,-0.845668,-1.003368,...,-0.516048,-0.349816,1.404233,-0.356693,0.033418,0.984988,-0.550601,1,0,1
2,0.911490,-0.420196,1.133103,0.117337,1.119048,0.354263,-0.130157,-0.464515,-0.035829,0.135960,...,-0.213967,-0.058566,0.128133,-0.364484,0.198135,0.136944,-0.317772,0,0,1
3,1.030213,0.059600,-0.052372,-0.529214,-0.947617,-0.382281,0.145189,-0.625161,-0.020178,-0.104260,...,0.096864,-0.113360,-0.079478,0.173965,-0.399200,0.685758,0.156323,1,0,0
4,-0.718215,-0.525791,-0.381046,-1.175765,-0.640684,-0.177539,0.619827,0.357343,0.244570,-0.539553,...,-0.689237,-0.513730,0.810782,0.737699,-0.920907,1.023206,-0.829647,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
128,-0.138070,1.544893,-2.124341,0.976326,-2.065485,0.042358,1.599456,0.375568,1.899515,-0.363672,...,0.930067,0.120148,0.964789,1.485388,-0.369031,-1.536947,0.730646,0,1,1
129,0.412098,0.576206,0.265743,-0.519977,-0.740064,-0.297368,0.558702,-0.298485,0.843780,-0.328021,...,0.703497,0.633593,0.292806,0.678069,0.414328,-1.779933,1.214287,0,0,1
130,0.106515,-0.387562,0.820833,-0.686233,1.037112,-1.686622,-1.329512,-0.033389,-0.554985,-0.940474,...,-0.450516,-0.226312,0.360834,-1.291783,0.221589,-1.328080,-0.287311,0,0,0
131,1.223753,-0.363854,1.379431,-1.351257,0.587048,-0.315683,-0.574494,-0.629875,0.050966,-0.224861,...,-0.409780,-0.106407,-0.039800,-0.756417,0.008580,-0.508439,-0.418102,0,0,1


### Predicting using Random Forest Regressor
- During training and evaluation, the model was fitted on values for RFS transformed using a logarithmic function
- Thus, the predictions need to be converted back into a normal scale using an exponential function

In [13]:
rf_reg = joblib.load('../exports/rfs_random_forest.pkl')

In [15]:
log_predictions = rf_reg.predict(Xs)
predictions = np.expm1(log_predictions)

In [20]:
#print(predictions)

In [21]:
with open("../datasets/RFSPrediction.csv", "w", encoding = "utf-8", newline = '') as file:
    write = pd.concat([ids, pd.DataFrame(predictions, columns = ['RelapseFreeSurvival (outcome)'])], ignore_index = False, sort = False, axis = 1)
    write.to_csv(file, index = False)

write

,ID,RelapseFreeSurvival (outcome)
0,TRG002219,63.011051
1,TRG002222,68.224967
2,TRG002223,69.314133
3,TRG002235,50.997049
4,TRG002240,58.352785
...,...,...
128,TRG002953,33.680487
129,TRG002955,45.713315
130,TRG002956,54.987062
131,TRG002957,44.593212
